# Micro-Capstone — Assistente de Conhecimento Interno (Porto Ventura Terminais)

Empresa **fictícia** de logística portuária. O assistente responde perguntas dos funcionários usando só as políticas internas em `base_conhecimento/`.

Este notebook segue o mesmo roteiro da aula M3D4 (Qwen2.5-1.5B-Instruct em 4 bits + LoRA com `SFTTrainer`), trocando a tarefa de sentimento pela de perguntas e respostas sobre a base. São comparados 4 modos no conjunto de teste:

| Modo | Modelo | Contexto (grounding) |
|---|---|---|
| A | base | não |
| B | base | sim (trechos recuperados por BM25) |
| C | base + LoRA | não |
| D | base + LoRA | sim |

**Ambiente:** Colab com GPU (Runtime → Change runtime type → T4 GPU).

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes

## 1. Código e base de conhecimento

Clona o repositório do projeto. Se o repositório estiver privado, faça upload do zip do projeto e descompacte na pasta `posIAM03D04`.

In [ ]:
import os, subprocess, sys

if not os.path.exists("posIAM03D04"):
    subprocess.run(["git", "clone", "-q", "https://github.com/lucianafm9/posIAM03D04.git"], check=True)

os.chdir("posIAM03D04")
sys.path.insert(0, os.getcwd())

from assistente.base import carregar_base
from assistente.busca import buscar
from assistente.dataset import K_TRECHOS, carregar_perguntas, gerar
from assistente.prompts import montar_mensagens, RECUSA

trechos = carregar_base()
print(f"{len(trechos)} trechos em {len({t.doc_id for t in trechos})} documentos")
for t in trechos[:5]:
    print(" -", t.referencia)

### Estrutura de contexto

Para cada pergunta, a busca BM25 recupera os 3 trechos mais relevantes. Os trechos entram no prompt identificados por `[POL-NN — documento — seção]`, e o prompt de sistema obriga o modelo a responder só com o contexto, citar a fonte e recusar quando a informação não estiver lá.

In [ ]:
pergunta = "Meu motorista chegou 40 minutos depois do fim da janela. Ele ainda pode entrar?"
for trecho, score in buscar(pergunta, K_TRECHOS):
    print(f"{score:5.2f}  {trecho.referencia}")

for m in montar_mensagens(pergunta, [t for t, _ in buscar(pergunta, K_TRECHOS)]):
    print(f"--- {m['role']} ---\n{m['content']}\n")

## 2. Modelo base (mesma configuração da aula)

In [ ]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

model.eval()

A função `ask` é a da aula com dois ajustes: o papel `system` estava escrito `sytem` (o modelo ignorava a instrução de sistema) e os tensores de entrada agora vão para o mesmo dispositivo do modelo.

In [ ]:
def ask(_model, tok, pergunta: str, com_contexto: bool) -> str:
  trechos = [t for t, _ in buscar(pergunta, K_TRECHOS)] if com_contexto else None
  messages = montar_mensagens(pergunta, trechos)

  inputs = tok.apply_chat_template(
      messages,
      add_generation_prompt=True,
      return_tensors="pt",
      return_dict=True
  ).to(_model.device)

  with torch.no_grad():
    output = _model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tok.eos_token_id,
    )

  return tok.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

### Baseline: com e sem grounding

Antes do fine-tuning, a mesma pergunta com e sem o contexto da base. Sem contexto, o modelo não conhece as regras do Porto Ventura e tende a inventar uma resposta plausível.

In [ ]:
baseline_perguntas = [
    "Qual a tolerância de atraso para o motorista agendado no gate?",
    "Quantos dias de free time tem um contêiner de importação?",
    "Quantos dias de licença-paternidade a empresa concede?",  # fora da base
]

for p in baseline_perguntas:
  print(f"Pergunta: {p}")
  print(f"  Sem contexto: {ask(model, tokenizer, p, com_contexto=False)}")
  print(f"  Com contexto: {ask(model, tokenizer, p, com_contexto=True)}\n")

## 3. Dataset de treino

`data/perguntas.json` tem 60 perguntas com gabarito: 40 de treino (30 na base e 10 fora da base) e 20 de teste (14 na base e 6 fora). Cada exemplo de treino vai no formato `messages` da aula: o `user` leva o contexto recuperado + a pergunta, e o `assistant` leva a resposta com a fonte, ou a frase de recusa quando a pergunta não está na base.

In [ ]:
from datasets import load_dataset

gerar()  # gera data/treino.jsonl e data/teste.jsonl
dataset_chat = load_dataset("json", data_files="data/treino.jsonl", split="train").remove_columns(["id"])
print(dataset_chat)
dataset_chat[0]["messages"]

## 4. Fine-tuning com LoRA (mesmos hiperparâmetros da aula)

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_chat,
    args=SFTConfig(
        output_dir="./lora-output",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        learning_rate=2e-4,
        logging_steps=5,
        report_to="none",
    )
)

trainer.train()
trainer.save_model("./lora-adapter-conhecimento")

In [ ]:
model.gradient_checkpointing_disable()
model.config.use_cache = True

model.eval()

## 5. Avaliação dos 4 modos no conjunto de teste

Os modos A e B usam `model.disable_adapter()`, como na função `compare` da aula. A avaliação é automática:

- pergunta na base: **correta** se a resposta traz todos os fatos do gabarito, **incompleta** se traz parte, **incorreta** se não traz nenhum e **recusa indevida** se recusou;
- pergunta fora da base: **recusa correta** se recusou, **alucinação** se inventou uma resposta.

A taxa de alucinação soma as respostas inventadas para perguntas fora da base e as respostas sem nenhum fato correto.

In [ ]:
import json
from assistente.avaliacao import avaliar, resumir

test_cases = [json.loads(l) for l in open("data/teste.jsonl", encoding="utf-8")]

def rodar(com_adapter: bool, com_contexto: bool) -> list[dict]:
  saida = []
  for caso in test_cases:
    if com_adapter:
      resposta = ask(model, tokenizer, caso["pergunta"], com_contexto)
    else:
      with model.disable_adapter():
        resposta = ask(model, tokenizer, caso["pergunta"], com_contexto)
    saida.append({"id": caso["id"], "pergunta": caso["pergunta"], "fonte": caso["fonte"],
                  "resposta": resposta, "avaliacao": avaliar(resposta, caso)})
  return saida

modos = {
    "base_sem_contexto": dict(com_adapter=False, com_contexto=False),
    "base_com_contexto": dict(com_adapter=False, com_contexto=True),
    "lora_sem_contexto": dict(com_adapter=True, com_contexto=False),
    "lora_com_contexto": dict(com_adapter=True, com_contexto=True),
}

respostas, metricas = {}, {}
for nome, cfg in modos.items():
  respostas[nome] = rodar(**cfg)
  metricas[nome] = resumir([r["avaliacao"] for r in respostas[nome]], test_cases)
  print(nome, metricas[nome])

In [ ]:
from IPython.display import Markdown, display
from assistente.relatorio import gerar_relatorio

os.makedirs("results", exist_ok=True)
json.dump(respostas, open("results/respostas.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump(metricas, open("results/metricas.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

display(Markdown(gerar_relatorio()))

## 6. Comparação lado a lado

In [ ]:
def compare(caso_id: str):
  caso = next(c for c in test_cases if c["id"] == caso_id)
  print(f"Pergunta: {caso['pergunta']}  (fonte esperada: {caso['fonte'] or 'fora da base'})")
  for nome in modos:
    r = next(x for x in respostas[nome] if x["id"] == caso_id)
    print(f"  {nome:18} [{r['avaliacao']['categoria']}] {r['resposta']}")
  print()

for caso_id in ["A01", "A09", "A14", "A15", "A18"]:
  compare(caso_id)

## 7. Download dos resultados e do adapter

Baixe o zip e copie a pasta `results/` para o repositório (são os arquivos usados no README).

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("resultados-d04", "zip", ".", "results")
shutil.make_archive("lora-adapter-conhecimento", "zip", "./lora-adapter-conhecimento")
files.download("resultados-d04.zip")
files.download("lora-adapter-conhecimento.zip")